In [0]:
%python
from pyspark.sql.functions import col, sum, count, round

sales = spark.table("workspace.rp_silver.sales_daily")
stores = spark.table("workspace.rp_silver.stores")
products = spark.table("workspace.rp_silver.products")

# Join - Sales + Stores + Products
joined = sales.join(stores, "store_id", "left").join(products, "product_id", "left")

# Gold 1 - Daily Summary
daily = joined.groupBy("sale_date").agg(
    count("*").alias("total_orders"),
    sum("qty").alias("total_qty"),
    round(sum("amount"), 2).alias("total_revenue")
).orderBy("sale_date")

daily.write.mode("overwrite").saveAsTable("workspace.rp_gold.daily_sales_summary")

# Gold 2 - Store Wise
store_wise = joined.groupBy("store_id", "store_name", "city").agg(
    count("*").alias("total_orders"),
    round(sum("amount"), 2).alias("total_revenue")
).orderBy(col("total_revenue").desc())

store_wise.write.mode("overwrite").saveAsTable("workspace.rp_gold.store_wise_sales")

print("M4 DONE - Gold tables created")
daily.show(5)
store_wise.show()